In [ ]:
# Setup the Jupyter version
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module
from Shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Local MongoDB Configuration
USER = ""           
PASS = ""           
HOST = "localhost"  # Local MongoDB
PORT = 27017        # Default MongoDB port
DB = "AAC"
COL = "animals"

# Connect to database via CRUD 
db = AnimalShelter(USER, PASS, HOST, PORT, DB, COL)

# Read all documents from database
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB '_id' column 
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

print(f"Loaded {len(df)} animals from database")

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load Grazioso Salvare logo
try:
    image_filename = "Logo.png"
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
    logo_exists = True
except:
    logo_exists = False
    print("Logo.png not found - continuing without logo")

# Dashboard Layout
app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    
    # Header
    html.Div(style={
        'backgroundColor': '#2C3E50',
        'color': 'white',
        'padding': '20px',
        'textAlign': 'center',
        'marginBottom': '20px'
    }, children=[
        html.H1('Grazioso Salvare Animal Rescue Dashboard', 
                style={'margin': '0'}),
        html.P('CS-340 Project 2 - Christina Jimenez',
               style={'margin': '10px 0 0 0'})
    ]),
    
    # Logo section
    html.Div(style={'textAlign': 'center', 'marginBottom': '20px'}, children=[
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '150px', 'margin': '10px'}) if logo_exists else html.Div(),
    ]),
    
    html.Hr(),
    
    # Filter section
    html.Div(style={
        'backgroundColor': '#ECF0F1',
        'padding': '20px',
        'borderRadius': '5px',
        'marginBottom': '20px'
    }, children=[
        html.H3('Filter by Rescue Type', style={'color': '#2C3E50'}),
        dcc.Dropdown(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster/Tracking', 'value': 'Disaster'},
                {'label': 'No Filter (Show All)', 'value': 'Reset'}
            ],
            value='Reset',
            clearable=False,
            style={'width': '100%'}
        )
    ]),
    
    html.Hr(),
    
    # Data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{
            "name": i.replace('_', ' ').title(),
            "id": i,
            "deletable": False,
            "selectable": True
        } for i in df.columns],
        data=df.to_dict('records'),
        row_selectable='single',
        selected_rows=[0],
        page_size=15,
        page_action='native',
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        style_table={
            'overflowX': 'auto',
            'maxHeight': '500px',
            'overflowY': 'auto'
        },
        style_cell={
            'textAlign': 'left',
            'padding': '10px',
            'fontFamily': 'Arial'
        },
        style_header={
            'backgroundColor': '#34495E',
            'color': 'white',
            'fontWeight': 'bold'
        },
        style_data_conditional=[
            {
                'if': {'row_index': 'odd'},
                'backgroundColor': '#F8F9F9'
            },
            {
                'if': {'state': 'selected'},
                'backgroundColor': '#3498DB',
                'color': 'white'
            }
        ]
    ),
    
    html.Br(),
    html.Hr(),
    
    # Visualizations side by side
    html.Div(className='row', style={'display': 'flex', 'gap': '20px'}, children=[
        html.Div(id='graph-id', style={'flex': '1', 'minWidth': '400px'}),
        html.Div(id='map-id', style={'flex': '1', 'minWidth': '400px'})
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """Update data table based on selected rescue type filter"""
    
    query = {}
    
    if filter_type == 'Water':
        # Water rescue criteria
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }
    
    elif filter_type == 'Mountain':
        # Mountain/wilderness rescue criteria
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", 
                       "Siberian Husky", "Rottweiler"]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }
    
    elif filter_type == 'Disaster':
        # Disaster/tracking rescue criteria
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", 
                       "Bloodhound", "Rottweiler"]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }
    
    else:  # Reset 
        query = {}
    
    # Fetch filtered data from database
    filtered_data = db.read(query)
    df_filtered = pd.DataFrame.from_records(filtered_data)
    
    # Remove '_id' column if present
    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)
    
    return df_filtered.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    """Update pie chart based on current data table view"""
    
    if viewData is None or len(viewData) == 0:
        return [html.P('No data to display', style={'textAlign': 'center', 'padding': '20px'})]
    
    df_graph = pd.DataFrame(viewData)
    
    # Count breeds
    if 'breed' not in df_graph.columns:
        return [html.P('No breed data available')]
    
    breed_counts = df_graph['breed'].value_counts().head(10)
    
    # Create pie chart
    figure = px.pie(
        values=breed_counts.values,
        names=breed_counts.index,
        title='Top 10 Breed Distribution',
        hole=0.3
    )
    
    figure.update_layout(
        showlegend=True,
        legend=dict(
            title="Breeds",
            orientation="v",
            x=1.05,
            y=1
        ),
        height=500
    )
    
    return [dcc.Graph(figure=figure)]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    """Highlight selected columns in data table"""
    
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    """Update map based on selected animal"""
    
    if viewData is None or not viewData or index is None or not index:
        return [
            html.P('Select a row to view location', 
                  style={'textAlign': 'center', 'padding': '20px'})
        ]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Get selected row (default to first row if none selected)
    row = index[0] if index else 0
    
    # Safety check
    if row >= len(dff):
        row = 0
    
    # Get location data
    try:
        lat = dff.iloc[row]['location_lat'] if 'location_lat' in dff.columns else 30.75
        lon = dff.iloc[row]['location_long'] if 'location_long' in dff.columns else -97.48
        breed = dff.iloc[row]['breed'] if 'breed' in dff.columns else 'Unknown'
        name = dff.iloc[row]['name'] if 'name' in dff.columns else 'Unknown'
    except:
        lat, lon = 30.75, -97.48
        breed, name = 'Unknown', 'Unknown'
    
    # Create map
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[lat, lon],
            zoom=12,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H4("Animal Details"),
                            html.P(f"Name: {name}"),
                            html.P(f"Breed: {breed}")
                        ])
                    ]
                )
            ]
        )
    ]


# Run the app
if __name__ == '__main__':
    print("\nStarting Grazioso Salvare Dashboard...")
    print("Access at: http://localhost:8050")
    print("Press CTRL+C to stop\n")
    app.run_server(debug=True, port=8050)

Dash app running on http://127.0.0.1:13540/


#### 